# 🎓 Aprendiendo Metaflow con las Reseñas del Profesor Raúl

**Metaflow** es un framework de MLOps creado por Netflix que nos permite construir, gestionar y desplegar pipelines de machine learning de forma **reproducible**, **escalable** y **organizada**.

En este notebook aprenderemos los conceptos clave de Metaflow construyendo un clasificador de sentimientos para las reseñas del Profesor Raúl. El modelo predecirá si una reseña es:
- **-1** → Negativa 😡
- **0** → Neutral 😐
- **1** → Positiva 😊

---

## 📚 Índice de conceptos que aprenderemos

| # | Concepto | Descripción |
|---|----------|-------------|
| 1 | `FlowSpec` | La clase base de todo flow en Metaflow |
| 2 | `@step` | Decorador para definir pasos del pipeline |
| 3 | `self` como almacén de datos | Cómo pasar datos entre pasos |
| 4 | `@card` | Visualizaciones automáticas |
| 5 | `@catch` | Manejo de errores robusto |
| 6 | `@retry` | Reintentos automáticos |
| 7 | `@timeout` | Límites de tiempo |
| 8 | Ramificación y unión (`fanout/join`) | Paralelismo en pasos |
| 9 | `@parameter` | Hiperparámetros configurables |
| 10 | `@conda` / `@pypi` | Gestión de dependencias |
| 11 | `current` | Contexto de ejecución |
| 12 | El Client API | Acceder a resultados de runs anteriores |

## 🛠️ Instalación y configuración

Primero instalamos Metaflow y las librerías necesarias.

In [1]:
# Instalamos las dependencias necesarias
!pip install metaflow scikit-learn pandas numpy matplotlib seaborn --quiet

print("✅ Instalación completada")

⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⣀⣀⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⣀⣴⣶⣶⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⣾⠟⠉⠉⠙⠻⣶⣄⠀⠀⠀⠀⠀⠀⠀⠀⢀⡀⠀⠀⠀⠀⠀⢀⣴⠟⠋⠀⠀⠀⠙⣷⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⣾⠃⠀⣠⡤⢄⡀⠀⠙⢷⣄⢀⣀⣀⣀⣾⢦⡿⣡⡶⣂⣀⣀⣴⠟⠁⠀⣠⠖⢋⠶⡀⢹⡇⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣇⠀⢰⠇⣽⣀⡻⣦⠀⠈⠛⠛⠋⠉⠉⠀⠀⠀⠀⠀⠈⠉⠉⠁⠀⠀⢼⣭⣞⣇⣴⡇⢸⡇⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⡇⠀⢸⡃⢠⢷⡱⠟⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠻⢽⣴⡇⢸⡇⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣇⠀⠘⢶⠟⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠀⣾⠃⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⣌⡿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠠⡆⠀⠈⠓⣄⠀⠀⠀⠀⡴⠁⠀⠀⠖⠒⠒⠢⢤⡀⠀⠙⣿⡛⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢨⣿⠇⠀⠀⠀⢀⡠⠎⠀⠀⠀⢀⣀⣀⣀⡀⠀⠘⣶⠾⠿⣶⡇⠀⣀⣤⣤⣄⣀⡀⠀⠙⠲⡄⢹⣟⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣰⡟⠀⠀⠀⡤⠋⠀⠀⣠⡴⠚⠏⠉⠉⠙⠉⠢⣤⡇⠀⠀⠈⢧⡾⠋⠁⠀⡀⠀⠋⢷⣄⠀⠘⢦⢻⣆⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⡿⠁⢀⡤⠊⠀⢀⣴⠾⠋⠀⠀⢀⣤⣤⣄⠀⠀⠉⠀⠀⠀⠀⠀⠀⠀⣠⣶⣶⡄⠀⠀⠛⢿⣄⠀⠑⢽⣧⣀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⣶⠟⠔⠒⠉⠀⠀⣰⠟⠁⠀⠀⠀⠀⣾⣿⣿⣿⡆⣀⣤⣤⡄⠀⡀⣤⣤⣶⣿⣿⣿⣿⠀⠀⠀⠀⠻⣧⠀⠀⢨⣍⠁⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣾⠃⠀⠀⠀⠀⣰⠃⠀⠀⠀⠀⠀⠀⠙⠿⠟⣁⠞⠉⠀⠀⣿⣿⣿⣿⡆⠀⠙⣿⠋⠁⠀⠀⠀⠀⠀⢻⡇⠀⠀⣽⠃⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢰⡄⠀⠀⠀⠀⢻⣇⠀⠀⠀⠀⠀⠀⠀⢀⡼⠋⠀⠀⠀⠀⠉⠛⣿⠋⠀⠀⣀⠈⠻⢄⡀⠀⠀⢀⡰⠏⠀⠀⢠⡿⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⣷⡀⠀⠀⠀⠀⠉⠳⠤⢤⣤⡤⠤⠚⠁⠀⠀⣸⣷⣦⣤⣴⠾⠻⠶⠶⠾⠛⠷⣦⡀⠀⠉⠉⠁⠀⠀⠀⣠⡿⠁⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⣼⠟⠁⣀⣀⠀⠀⠀⢶⠄⠀⢀⣤⡌⠻⣧⡀⠀⠀⢀⣠⡾⠟⠁⠀⠀⠀⠀⠀
⠀⠀⣤⣴⣾⠿⠒⠳⢶⣤⡀⠀⠀⢨⣿⠛⠦⠤⠀⠀⠀⠀⠀⠀⠀⠀⣾⠃⠀⠀⠿⠿⠀⠀⠀⠀⠀⠀⠈⠛⠃⢀⣸⣧⣀⠰⠛⣿⠗⠀⠀⠀⠀⠀⠀⠀
⠀⣴⡟⠉⠀⠀⠀⢀⠀⠉⢿⣄⠀⠐⣿⠂⠀⠀⠀⠀⠀⣀⣀⣀⣤⡾⠟⣛⣻⣿⣄⠀⠀⠀⠇⠀⠀⣠⣄⠀⢠⡿⠿⡯⠻⢷⣄⣙⢷⡄⠀⠀⠀⠀⠀⠀
⢸⣏⠃⠀⠀⠀⣠⠖⠶⠞⠋⣿⣀⣾⠏⠀⠀⠀⠀⠀⠈⠉⠉⢻⡋⠀⠀⠈⢩⣴⣿

## 📊 Creamos el dataset del Profesor Raúl

Generamos un CSV de ejemplo con reseñas y sus categorías. En producción, este archivo ya existiría.

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("profesor_raul_reviews.csv")
print(f"✅ CSV cargado: {len(df)} reseñas")
print(f"\nDistribución de categorías:")
print(df["category"].value_counts().sort_index())
print(f"\nPrimeras filas:")
df.head()


✅ CSV cargado: 5000 reseñas

Distribución de categorías:
category
-1    2000
 0    1000
 1    2000
Name: count, dtype: int64

Primeras filas:


,review,category
0,Raul is incredibly rude when you go to office ...,-1
1,The lectures are well-structured and Raul keep...,1
2,Raul makes even the most boring topics sound f...,1
3,I struggled to stay awake. Raul's teaching sty...,-1
4,"The class is fine, Raul stays on topic most of...",0


---

## 🧠 Concepto 1: `FlowSpec` — La clase base de Metaflow

Todo pipeline en Metaflow es una **clase Python** que hereda de `FlowSpec`.
Esta clase le indica a Metaflow que estamos definiendo un flujo de trabajo.

```
FlowSpec
  └── Tu clase (ej: ProfesorRaulFlow)
        ├── step start
        ├── step procesar_datos
        ├── step entrenar
        └── step end
```

---

## 🧠 Concepto 2: `@step` — Los pasos del pipeline

Cada método decorado con `@step` es un **nodo del grafo de ejecución**.
Metaflow garantiza que los pasos se ejecutan en orden y que los datos se persisten entre ellos.

**Reglas importantes:**
- El primer paso **siempre** se llama `start`
- El último paso **siempre** se llama `end`
- Cada paso llama a `self.next(...)` para indicar el siguiente

---

## 🧠 Concepto 3: `self` como almacén de datos entre pasos

Los atributos guardados en `self` dentro de un paso están disponibles en **todos los pasos siguientes**. Metaflow los serializa y persiste automáticamente.

```python
# En el paso A:
self.datos = pd.read_csv("archivo.csv")  # Guardamos

# En el paso B (siguiente):
print(self.datos.shape)  # ✅ Disponible aquí
```

---

## 🧠 Concepto 4: `Parameter` — Hiperparámetros configurables

Los `Parameter` permiten configurar el flow desde la línea de comandos o al ejecutarlo, sin modificar el código fuente.

```python
from metaflow import Parameter

class MiFlow(FlowSpec):
    test_size = Parameter('test_size', default=0.2, help='Proporción de test')
```

---

## 🧠 Concepto 5: Ramificación y unión (`fanout` / `join`)

Metaflow permite ejecutar varios pasos **en paralelo** y luego unirlos:

```
start → procesar → [entrenar_lr, entrenar_rf, entrenar_svm] → comparar → end
                    (paralelo)                                 (join)
```

---

## 🧠 Concepto 6: `@catch`, `@retry`, `@timeout` — Resiliencia

Metaflow tiene decoradores para hacer los pipelines robustos:
- **`@catch`**: Si el paso falla, continúa con un valor por defecto
- **`@retry`**: Reintenta el paso automáticamente N veces
- **`@timeout`**: Cancela el paso si tarda demasiado

---

## 🧠 Concepto 7: `@card` — Visualizaciones automáticas

Las cards son **informes HTML** que Metaflow genera automáticamente al ejecutar un flow. Muy útiles para monitorizar resultados.

---

## 🧠 Concepto 8: `current` — El contexto de ejecución

El objeto `current` de Metaflow nos da información sobre la ejecución actual:
- `current.run_id` — ID único de esta ejecución
- `current.flow_name` — Nombre del flow
- `current.step_name` — Nombre del paso actual
- `current.task_id` — ID de la tarea

---

Ahora construyamos el flow completo que integra **todos** estos conceptos:

In [4]:
# Escribimos el flow en un archivo .py para poder ejecutarlo
flow_code = '''
# =============================================================================
# FLOW COMPLETO DEL PROFESOR RAÚL
# Demuestra todos los conceptos clave de Metaflow
# =============================================================================

from metaflow import (
    FlowSpec, step, Parameter, current,
    card, catch, retry, timeout
)
from metaflow.cards import Table, Markdown, Image
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib
matplotlib.use("Agg")  # Backend sin pantalla para entornos no interactivos
import matplotlib.pyplot as plt
import io, base64


class ProfesorRaulFlow(FlowSpec):
    """
    Pipeline de clasificación de reseñas del Profesor Raúl.

    Clasifica reseñas en:
      -1 = Negativa
       0 = Neutral
       1 = Positiva

    Arquitectura del grafo:

        start
          |
        cargar_datos
          |
        preprocesar
          |
        ┌─────────────────────┐
        │         │           │
     entrenar_lr  entrenar_rf  entrenar_svm
        │         │           │
        └─────────┴───────────┘
                  │
              comparar_modelos
                  │
                 end
    """

    # =========================================================================
    # CONCEPTO: Parameter
    # Los Parameters son hiperparámetros configurables desde la línea de
    # comandos. Permiten experimentar sin tocar el código.
    # Uso: python flow.py run --test_size 0.3 --max_features 2000
    # =========================================================================

    ruta_csv = Parameter(
        "ruta_csv",
        default="profesor_raul_reviews.csv",
        help="Ruta al archivo CSV con las reseñas",
    )

    test_size = Parameter(
        "test_size",
        default=0.25,
        help="Proporción de datos para el conjunto de test (entre 0 y 1)",
    )

    max_features = Parameter(
        "max_features",
        default=1000,
        help="Número máximo de características TF-IDF",
    )

    random_seed = Parameter(
        "random_seed",
        default=42,
        help="Semilla aleatoria para reproducibilidad",
    )

    # =========================================================================
    # PASO 1: start
    # El primer paso de TODOS los flows se llama obligatoriamente "start".
    # Es el punto de entrada del pipeline.
    # =========================================================================

    @step
    def start(self):
        """
        Punto de entrada del flow.
        Imprimimos información de contexto usando el objeto `current`.

        CONCEPTO: current
        El objeto `current` nos da acceso al contexto de la ejecución:
        - current.flow_name  → nombre de la clase del flow
        - current.run_id     → ID único de esta ejecución
        - current.step_name  → nombre del paso actual
        - current.task_id    → ID de la tarea dentro del paso

        Esto es muy útil para logging, debugging y trazabilidad.
        """
        print("=" * 60)
        print(f"🚀 Iniciando flow: {current.flow_name}")
        print(f"   Run ID      : {current.run_id}")
        print(f"   Paso actual : {current.step_name}")
        print(f"   Task ID     : {current.task_id}")
        print("=" * 60)
        print(f"\nParámetros configurados:")
        print(f"  - CSV         : {self.ruta_csv}")
        print(f"  - Test size   : {self.test_size}")
        print(f"  - Max features: {self.max_features}")
        print(f"  - Random seed : {self.random_seed}")

        # self.next() indica cuál es el SIGUIENTE PASO en el grafo
        self.next(self.cargar_datos)

    # =========================================================================
    # PASO 2: cargar_datos
    # Cargamos el CSV y validamos su estructura.
    #
    # CONCEPTO: @catch
    # Si este paso lanza una excepción, Metaflow la captura y continúa.
    # El atributo `exception` guardará el error para inspeccionarlo después.
    # =========================================================================

    @catch(var="error_carga")  # Si falla, guarda la excepción en self.error_carga
    @retry(times=2)            # Si falla, reintenta hasta 2 veces
    @timeout(seconds=30)       # Si tarda más de 30s, cancela
    @step
    def cargar_datos(self):
        """
        Carga el CSV y valida que tenga las columnas correctas.

        CONCEPTO: @catch, @retry, @timeout
        - @catch: captura errores sin detener el pipeline
        - @retry: reintenta el paso si falla (útil para I/O con red)
        - @timeout: evita que un paso se quede colgado indefinidamente

        El orden de los decoradores importa: se aplican de abajo hacia arriba.

        CONCEPTO: self como almacén de datos
        Todo lo que guardamos en `self` estará disponible en los pasos
        siguientes. Metaflow serializa estos atributos automáticamente
        usando su propio sistema de almacenamiento (datastore).
        """
        print(f"\n📂 Cargando datos desde: {self.ruta_csv}")

        # Cargamos el CSV — si falla, @catch lo captura
        df = pd.read_csv(self.ruta_csv)

        # Validación básica de la estructura
        columnas_esperadas = {"review", "category"}
        if not columnas_esperadas.issubset(df.columns):
            raise ValueError(
                f"El CSV debe tener columnas {columnas_esperadas}. "
                f"Encontradas: {set(df.columns)}"
            )

        categorias_validas = {-1, 0, 1}
        categorias_encontradas = set(df["category"].unique())
        if not categorias_encontradas.issubset(categorias_validas):
            raise ValueError(
                f"Categorías inválidas: {categorias_encontradas - categorias_validas}"
            )

        # Eliminamos filas con valores nulos
        df = df.dropna(subset=["review", "category"])
        df["category"] = df["category"].astype(int)

        # GUARDAMOS en self → disponible en todos los pasos siguientes
        self.df = df
        self.n_muestras = len(df)
        self.distribucion = df["category"].value_counts().sort_index().to_dict()

        print(f"✅ Dataset cargado: {self.n_muestras} reseñas")
        print(f"   Distribución: {self.distribucion}")

        self.next(self.preprocesar)

    # =========================================================================
    # PASO 3: preprocesar
    # Dividimos datos y vectorizamos el texto con TF-IDF.
    # =========================================================================

    @step
    def preprocesar(self):
        """
        Preprocesamiento del texto y división train/test.

        Usamos TF-IDF (Term Frequency - Inverse Document Frequency) para
        convertir el texto en vectores numéricos que los modelos puedan usar.

        TF-IDF asigna mayor peso a palabras que son:
        - Frecuentes en un documento (TF alto)
        - Raras en el corpus general (IDF alto)
        """
        print("\n⚙️  Preprocesando datos...")

        # Separamos features y etiquetas
        X = self.df["review"].values
        y = self.df["category"].values

        # División estratificada train/test
        # Estratificada = mantiene la proporción de clases en ambos conjuntos
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=self.test_size,
            random_state=self.random_seed,
            stratify=y,
        )

        # Vectorización TF-IDF con n-gramas (1 y 2 palabras)
        # Entrenamos el vectorizador SOLO en train para evitar data leakage
        vectorizador = TfidfVectorizer(
            max_features=self.max_features,
            ngram_range=(1, 2),    # Unigramas y bigramas
            strip_accents="unicode",
            min_df=1,              # Mínimo de documentos donde aparece el término
            sublinear_tf=True,     # Aplica log(1 + tf) para suavizar frecuencias
        )

        X_train_vec = vectorizador.fit_transform(X_train)  # Fit + transform en train
        X_test_vec  = vectorizador.transform(X_test)       # Solo transform en test

        # Guardamos todo en self para los pasos siguientes
        self.X_train_vec = X_train_vec
        self.X_test_vec  = X_test_vec
        self.y_train     = y_train
        self.y_test      = y_test
        self.vectorizador = vectorizador
        self.n_train = len(y_train)
        self.n_test  = len(y_test)

        print(f"✅ Preprocesamiento completado")
        print(f"   Train: {self.n_train} muestras")
        print(f"   Test : {self.n_test} muestras")
        print(f"   Vocabulario TF-IDF: {X_train_vec.shape[1]} términos")

        # CONCEPTO: Ramificación (fanout)
        # self.next() acepta MÚLTIPLES pasos → los ejecuta en PARALELO
        # Esto es una de las características más poderosas de Metaflow
        self.next(
            self.entrenar_regresion_logistica,
            self.entrenar_random_forest,
            self.entrenar_svm,
        )

    # =========================================================================
    # RAMAS PARALELAS: Tres modelos entrenándose a la vez
    # Cada uno guarda sus resultados en self, que luego el join recoge
    # =========================================================================

    @step
    def entrenar_regresion_logistica(self):
        """
        Rama 1: Regresión Logística.

        Modelo lineal, muy eficiente y buena línea base para NLP.
        Con C alto, menos regularización; con C bajo, más regularización.
        """
        print("\n🔵 [Rama 1] Entrenando Regresión Logística...")

        modelo = LogisticRegression(
            C=1.0,
            max_iter=1000,
            multi_class="multinomial",  # Clasificación multiclase
            random_state=self.random_seed,
        )
        modelo.fit(self.X_train_vec, self.y_train)

        predicciones = modelo.predict(self.X_test_vec)
        accuracy = accuracy_score(self.y_test, predicciones)

        # Guardamos resultados — el JOIN los recogerá
        self.nombre_modelo  = "Regresión Logística"
        self.modelo         = modelo
        self.predicciones   = predicciones
        self.accuracy       = accuracy
        self.reporte        = classification_report(
            self.y_test, predicciones, target_names=["Negativa", "Neutral", "Positiva"]
        )

        print(f"   ✅ Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
        self.next(self.comparar_modelos)

    @step
    def entrenar_random_forest(self):
        """
        Rama 2: Random Forest.

        Ensemble de árboles de decisión. Más robusto al overfitting
        que un solo árbol, pero más lento que la regresión logística.
        """
        print("\n🟢 [Rama 2] Entrenando Random Forest...")

        modelo = RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=2,
            random_state=self.random_seed,
            n_jobs=-1,  # Usa todos los CPUs disponibles
        )
        modelo.fit(self.X_train_vec, self.y_train)

        predicciones = modelo.predict(self.X_test_vec)
        accuracy = accuracy_score(self.y_test, predicciones)

        self.nombre_modelo  = "Random Forest"
        self.modelo         = modelo
        self.predicciones   = predicciones
        self.accuracy       = accuracy
        self.reporte        = classification_report(
            self.y_test, predicciones, target_names=["Negativa", "Neutral", "Positiva"]
        )

        print(f"   ✅ Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
        self.next(self.comparar_modelos)

    @step
    def entrenar_svm(self):
        """
        Rama 3: Support Vector Machine (LinearSVC).

        Los SVM son muy efectivos para clasificación de texto con TF-IDF.
        LinearSVC es una versión eficiente para datasets grandes.
        """
        print("\n🔴 [Rama 3] Entrenando SVM (LinearSVC)...")

        modelo = LinearSVC(
            C=1.0,
            max_iter=2000,
            random_state=self.random_seed,
        )
        modelo.fit(self.X_train_vec, self.y_train)

        predicciones = modelo.predict(self.X_test_vec)
        accuracy = accuracy_score(self.y_test, predicciones)

        self.nombre_modelo  = "SVM (LinearSVC)"
        self.modelo         = modelo
        self.predicciones   = predicciones
        self.accuracy       = accuracy
        self.reporte        = classification_report(
            self.y_test, predicciones, target_names=["Negativa", "Neutral", "Positiva"]
        )

        print(f"   ✅ Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
        self.next(self.comparar_modelos)

    # =========================================================================
    # JOIN: Recoge los resultados de todas las ramas paralelas
    # =========================================================================

    @card  # CONCEPTO: @card genera un informe HTML visual automáticamente
    @step
    def comparar_modelos(self, inputs):
        """
        CONCEPTO: Join (unión de ramas paralelas)

        Cuando múltiples pasos convergen en uno solo, el paso de unión
        recibe `inputs` — un iterable con los resultados de cada rama.

        Patrón habitual:
            for inp in inputs:
                # inp.nombre_modelo, inp.accuracy, etc.

        También usamos `self.merge_artifacts(inputs, ...)` para heredar
        atributos comunes de todas las ramas (los que tienen el mismo valor).

        CONCEPTO: @card
        El decorador @card le dice a Metaflow que genere un informe visual
        para este paso. Podemos personalizarlo con componentes como
        Table, Markdown, Image.
        """
        from metaflow.cards import Table, Markdown

        print("\n🏆 Comparando resultados de los modelos...")

        # Recopilamos resultados de todas las ramas
        resultados = []
        mejor_accuracy = -1
        mejor_modelo = None
        mejor_nombre = ""
        mejor_predicciones = None
        mejor_reporte = ""

        for inp in inputs:
            resultados.append({
                "Modelo"   : inp.nombre_modelo,
                "Accuracy" : f"{inp.accuracy:.4f}",
                "Accuracy%": f"{inp.accuracy*100:.1f}%",
            })
            if inp.accuracy > mejor_accuracy:
                mejor_accuracy    = inp.accuracy
                mejor_modelo      = inp.modelo
                mejor_nombre      = inp.nombre_modelo
                mejor_predicciones = inp.predicciones
                mejor_reporte     = inp.reporte

        # Heredamos atributos comunes de las ramas (vectorizador, y_test, etc.)
        # merge_artifacts toma el valor de la primera rama que lo tenga,
        # siempre que sea el mismo en todas las ramas
        self.merge_artifacts(inputs, include=["vectorizador", "y_test"])

        # Guardamos los resultados finales
        self.resultados_modelos  = resultados
        self.mejor_modelo        = mejor_modelo
        self.mejor_nombre        = mejor_nombre
        self.mejor_accuracy      = mejor_accuracy
        self.mejor_predicciones  = mejor_predicciones
        self.mejor_reporte       = mejor_reporte

        print("\n📊 RESULTADOS FINALES:")
        print("-" * 45)
        for r in sorted(resultados, key=lambda x: x["Accuracy"], reverse=True):
            prefijo = "🥇" if r["Modelo"] == mejor_nombre else "  "
            print(f"{prefijo} {r['Modelo']:30s} → {r['Accuracy%']}")
        print("-" * 45)
        print(f"\n✅ Mejor modelo: {mejor_nombre} ({mejor_accuracy*100:.1f}%)")
        print(f"\nReporte detallado del mejor modelo:")
        print(mejor_reporte)

        self.next(self.end)

    # =========================================================================
    # PASO FINAL: end
    # El último paso SIEMPRE se llama "end".
    # =========================================================================

    @step
    def end(self):
        """
        Paso final obligatorio del flow.

        Aquí hacemos limpieza final, logging de resultados,
        o cualquier acción de cierre del pipeline.

        IMPORTANTE: El paso `end` NO llama a self.next().
        Es el nodo terminal del grafo.
        """
        print("\n" + "=" * 60)
        print(f"🎉 Flow completado exitosamente")
        print(f"   Flow     : {current.flow_name}")
        print(f"   Run ID   : {current.run_id}")
        print(f"   Mejor    : {self.mejor_nombre}")
        print(f"   Accuracy : {self.mejor_accuracy*100:.1f}%")
        print("=" * 60)
        print("\n💡 Para acceder a estos resultados más tarde, usa el Client API:")
        print(f"   from metaflow import Flow")
        print(f\'   run = Flow(\"ProfesorRaulFlow\").latest_run\')
        print(f"   print(run.data.mejor_nombre)")
        print(f"   print(run.data.mejor_accuracy)")


if __name__ == "__main__":
    ProfesorRaulFlow()
'''

with open("profesor_raul_flow.py", "w", encoding="utf-8") as f:
    f.write(flow_code)

print("✅ Flow escrito en: profesor_raul_flow.py")

✅ Flow escrito en: profesor_raul_flow.py


## ▶️ Ejecutar el Flow

Para ejecutar el flow, usa uno de estos comandos:

```bash
# Ejecución básica
python profesor_raul_flow.py run

# Con parámetros personalizados
python profesor_raul_flow.py run --test_size 0.3 --max_features 2000

# Ver el grafo del flow sin ejecutarlo
python profesor_raul_flow.py show

# Ver todos los runs anteriores
python profesor_raul_flow.py list
```

En este notebook ejecutamos el flow localmente:

In [12]:
# Ejecutamos el flow desde el notebook
# La flag --no-pylint evita errores de linting en el entorno del notebook
import subprocess, sys

resultado = subprocess.run(
    [sys.executable, "profesor_raul_flow.py", "--no-pylint", "run"],
    capture_output=True,
    text=True
)

print(resultado.stdout)
if resultado.returncode != 0:
    print("STDERR:", resultado.stderr[-3000:])

2026-05-08 10:51:35.675 Workflow starting (run-id 1778230295672955):
2026-05-08 10:51:35.695 [1778230295672955/start/1 (pid 27465)] Task is starting.
2026-05-08 10:51:37.027 [1778230295672955/start/1 (pid 27465)] ============================================================
2026-05-08 10:51:37.027 [1778230295672955/start/1 (pid 27465)] 🚀 Iniciando flow: ProfesorRaulFlow
2026-05-08 10:51:37.027 [1778230295672955/start/1 (pid 27465)] Run ID      : 1778230295672955
2026-05-08 10:51:37.027 [1778230295672955/start/1 (pid 27465)] Paso actual : start
2026-05-08 10:51:37.027 [1778230295672955/start/1 (pid 27465)] Task ID     : 1
2026-05-08 10:51:37.027 [1778230295672955/start/1 (pid 27465)] ============================================================
2026-05-08 10:51:37.027 [1778230295672955/start/1 (pid 27465)] Parámetros configurados:
2026-05-08 10:51:37.027 [1778230295672955/start/1 (pid 27465)] - CSV         : profesor_raul_reviews.csv
2026-05-08 10:51:37.027 [1778230295672955/start/1 (pid 

---

## 🧠 Concepto 9: El Client API — Acceder a resultados de runs anteriores

Una de las características más potentes de Metaflow es que **persiste todos los resultados** de cada ejecución. El **Client API** nos permite acceder a ellos en cualquier momento, desde cualquier notebook o script.

Esto es fundamental para:
- **Reproducibilidad**: siempre puedes volver a cualquier experimento pasado
- **Comparación de experimentos**: compara resultados entre diferentes runs
- **Auditoría**: saber exactamente qué modelo se usó en producción y cuándo

In [13]:
# =============================================================================
# CONCEPTO: Client API de Metaflow
# =============================================================================

from metaflow import Flow, Run, Task, Step

# --- Flow: accede a todos los runs de un flow ---
flow = Flow("ProfesorRaulFlow")
print(f"Flow: {flow}")
print(f"Número de runs ejecutados: {len(list(flow.runs()))}")

# --- Run: accede a una ejecución concreta ---
ultimo_run = flow.latest_run
print(f"\nÚltimo Run ID    : {ultimo_run.id}")
print(f"Estado          : {'✅ Exitoso' if ultimo_run.successful else '❌ Fallido'}")
print(f"Creado          : {ultimo_run.created_at}")

Flow: Flow('ProfesorRaulFlow')
Número de runs ejecutados: 3

Último Run ID    : 1778230295672955
Estado          : ✅ Exitoso
Creado          : 2026-05-08 10:51:35.674000


In [14]:
# Accedemos a los datos (artifacts) guardados en el run
datos = ultimo_run.data

print("\n📊 Resultados del último run:")
print(f"   Mejor modelo  : {datos.mejor_nombre}")
print(f"   Accuracy      : {datos.mejor_accuracy*100:.1f}%")
print(f"   Muestras train: {datos.n_train}")
print(f"   Muestras test : {datos.n_test}")

print(f"\n📋 Comparación de modelos:")
for r in sorted(datos.resultados_modelos, key=lambda x: x["Accuracy"], reverse=True):
    emoji = "🥇" if r["Modelo"] == datos.mejor_nombre else "  "
    print(f"   {emoji} {r['Modelo']:30s} → {r['Accuracy%']}")

print(f"\n📝 Reporte detallado:")
print(datos.mejor_reporte)


📊 Resultados del último run:
   Mejor modelo  : SVM (LinearSVC)
   Accuracy      : 100.0%
   Muestras train: 3750
   Muestras test : 1250

📋 Comparación de modelos:
   🥇 SVM (LinearSVC)                → 100.0%
      Random Forest                  → 100.0%
      Regresión Logística            → 100.0%

📝 Reporte detallado:
              precision    recall  f1-score   support

    Negativa       1.00      1.00      1.00       500
     Neutral       1.00      1.00      1.00       250
    Positiva       1.00      1.00      1.00       500

    accuracy                           1.00      1250
   macro avg       1.00      1.00      1.00      1250
weighted avg       1.00      1.00      1.00      1250



In [15]:
# Inspeccionamos los pasos del run
print("Duplicados exactos:", df.duplicated(subset="review").sum())
print("🔍 Pasos del run:")
for step in ultimo_run.steps():
    tareas = list(step.tasks())
    estados = ["✅" if t.successful else "❌" for t in tareas]
    print(f"   {' '.join(estados)} {step.id}")

Duplicados exactos: 4946
🔍 Pasos del run:
   ✅ end
   ✅ comparar_modelos
   ✅ entrenar_svm
   ✅ entrenar_random_forest
   ✅ entrenar_regresion_logistica
   ✅ preprocesar
   ✅ cargar_datos
   ✅ start


---

## 🧠 Concepto 10: Predicción con el modelo guardado

Gracias a que Metaflow serializa los artifacts, podemos recuperar el modelo entrenado y usarlo para hacer predicciones nuevas sin tener que reentrenarlo.

In [ ]:
# Recuperamos el modelo y vectorizador del run
modelo_guardado = datos.mejor_modelo
vectorizador_guardado = datos.vectorizador

# Función de predicción lista para usar
def predecir_sentimiento(reseña: str) -> dict:
    """Predice el sentimiento de una reseña del Profesor Raúl."""
    etiquetas = {-1: "😡 Negativa", 0: "😐 Neutral", 1: "😊 Positiva"}
    
    vec = vectorizador_guardado.transform([reseña])
    prediccion = modelo_guardado.predict(vec)[0]
    
    return {
        "reseña"    : reseña,
        "categoría" : prediccion,
        "sentimiento": etiquetas[prediccion],
    }

# Probamos con nuevas reseñas
nuevas_reseñas = [
    "El profesor Raúl es absolutamente increíble, sus clases son lo mejor de la carrera",
    "Las clases del profesor Raúl están bien pero tampoco es para tanto",
    "Raúl es un pésimo docente, nunca explica nada y los exámenes son injustos",
    "Apruebo sus asignaturas sin mucho esfuerzo, es un profe bastante normal",
]

print("🔮 Predicciones con el modelo guardado:")
print("=" * 70)
for reseña in nuevas_reseñas:
    resultado = predecir_sentimiento(reseña)
    print(f"{resultado['sentimiento']}  →  {resultado['reseña'][:65]}..." 
          if len(resultado['reseña']) > 65 else 
          f"{resultado['sentimiento']}  →  {resultado['reseña']}")
print("=" * 70)

---

## 📈 Visualización de resultados

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Resultados del Pipeline — Profesor Raúl Flow", fontsize=14, fontweight="bold")

# --- Gráfico 1: Comparación de modelos ---
ax1 = axes[0]
modelos  = [r["Modelo"] for r in datos.resultados_modelos]
accuracy = [float(r["Accuracy"]) for r in datos.resultados_modelos]
colores  = ["#2ecc71" if m == datos.mejor_nombre else "#3498db" for m in modelos]

bars = ax1.barh(modelos, [a * 100 for a in accuracy], color=colores, edgecolor="white", height=0.5)
ax1.set_xlabel("Accuracy (%)", fontsize=11)
ax1.set_title("Comparación de Modelos", fontsize=12, fontweight="bold")
ax1.set_xlim(0, 110)
ax1.axvline(x=100/3 * 2, color="gray", linestyle="--", alpha=0.5, label="Baseline 66.7%")

for bar, acc in zip(bars, accuracy):
    ax1.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f"{acc*100:.1f}%", va="center", fontweight="bold")

verde_patch = mpatches.Patch(color="#2ecc71", label=f"Mejor: {datos.mejor_nombre}")
azul_patch  = mpatches.Patch(color="#3498db", label="Otros modelos")
ax1.legend(handles=[verde_patch, azul_patch], loc="lower right")
ax1.grid(axis="x", alpha=0.3)

# --- Gráfico 2: Matriz de confusión del mejor modelo ---
ax2 = axes[1]
cm = confusion_matrix(datos.y_test, datos.mejor_predicciones)
etiquetas = ["Negativa (-1)", "Neutral (0)", "Positiva (1)"]

im = ax2.imshow(cm, interpolation="nearest", cmap="Blues")
plt.colorbar(im, ax=ax2)
ax2.set_xticks(range(3))
ax2.set_yticks(range(3))
ax2.set_xticklabels(etiquetas, rotation=20, ha="right", fontsize=9)
ax2.set_yticklabels(etiquetas, fontsize=9)
ax2.set_xlabel("Predicho", fontsize=11)
ax2.set_ylabel("Real", fontsize=11)
ax2.set_title(f"Matriz de Confusión\n({datos.mejor_nombre})", fontsize=12, fontweight="bold")

for i in range(3):
    for j in range(3):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax2.text(j, i, str(cm[i, j]), ha="center", va="center",
                 color=color, fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("resultados_profesor_raul.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Gráfico guardado en resultados_profesor_raul.png")

---

## 🧠 Concepto 11: `@conda` y `@pypi` — Gestión de dependencias

En producción o en la nube, cada paso puede tener sus **propias dependencias** encapsuladas. Esto garantiza reproducibilidad total.

```python
from metaflow import conda, pypi

class MiFlow(FlowSpec):

    @pypi(packages={"scikit-learn": "1.4.0", "pandas": "2.1.0"})
    @step
    def entrenar(self):
        # Este paso siempre usará scikit-learn 1.4.0 exactamente
        from sklearn.ensemble import RandomForestClassifier
        ...

    @conda(packages={"pytorch": "2.1.0"}, channels=["pytorch"])
    @step
    def inferencia_gpu(self):
        # Este paso usa conda para instalar PyTorch con GPU
        import torch
        ...
```

---

## 🧠 Concepto 12: Ejecutar en la nube con `@batch` o `@kubernetes`

Metaflow hace trivial escalar a la nube. Solo añadimos un decorador:

```python
from metaflow import batch, kubernetes

class MiFlow(FlowSpec):

    @batch(cpu=4, memory=8000, gpu=1)  # Ejecuta en AWS Batch
    @step
    def entrenar_en_nube(self):
        # Este paso corre en AWS con 4 CPUs, 8GB RAM y 1 GPU
        ...

    @kubernetes(cpu=2, memory=4096)  # Ejecuta en Kubernetes
    @step
    def prediccion_escalable(self):
        ...
```

El código local y en la nube es **idéntico**. Solo cambia el decorador.

---

## 📋 Resumen: Todos los conceptos de Metaflow

In [ ]:
resumen = """
╔══════════════════════════════════════════════════════════════════════════════╗
║             RESUMEN DE CONCEPTOS CLAVE DE METAFLOW                         ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  1. FlowSpec      Clase base de todo flow. Herédala para crear pipelines.   ║
║                                                                              ║
║  2. @step         Decora métodos para convertirlos en pasos del pipeline.   ║
║                   Siempre empieza en `start` y termina en `end`.            ║
║                                                                              ║
║  3. self          Almacén de datos entre pasos. Todo atributo guardado en   ║
║                   self es serializado y disponible en pasos siguientes.      ║
║                                                                              ║
║  4. Parameter     Hiperparámetros configurables desde CLI sin tocar código.  ║
║                                                                              ║
║  5. Fanout/Join   self.next(paso_a, paso_b) → ejecución paralela.           ║
║                   El join recibe `inputs` con resultados de cada rama.       ║
║                                                                              ║
║  6. @catch        Captura errores sin detener el pipeline.                  ║
║                                                                              ║
║  7. @retry        Reintenta un paso N veces si falla. Ideal para I/O.      ║
║                                                                              ║
║  8. @timeout      Cancela el paso si supera el límite de tiempo.            ║
║                                                                              ║
║  9. @card         Genera informes HTML visuales automáticamente.            ║
║                                                                              ║
║ 10. current       Contexto de ejecución: run_id, step_name, flow_name...    ║
║                                                                              ║
║ 11. Client API    Flow(), Run(), Step() → accede a runs pasados.            ║
║                   Fundamental para reproducibilidad y comparar experimentos. ║
║                                                                              ║
║ 12. @conda/@pypi  Dependencias encapsuladas por paso. Reproducibilidad      ║
║                   garantizada en cualquier entorno.                          ║
║                                                                              ║
║ 13. @batch/@k8s   Un decorador para escalar a AWS/Kubernetes. El código     ║
║                   local y en la nube es idéntico.                            ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
print(resumen)

---

## 🚀 Próximos pasos

Ahora que conoces los conceptos fundamentales de Metaflow, puedes:

1. **Mejorar el modelo**: prueba con `max_features=5000` o añade más datos al CSV
2. **Añadir más modelos** a las ramas paralelas (XGBoost, LightGBM, etc.)
3. **Desplegar en la nube**: configura AWS Batch y añade `@batch` a los pasos de entrenamiento
4. **Experimentar con parámetros**: ejecuta `python profesor_raul_flow.py run --max_features 3000 --test_size 0.2`
5. **Explorar el Client API**: compara la accuracy entre diferentes runs con el histórico de Metaflow

### 📚 Recursos oficiales
- [Documentación oficial de Metaflow](https://docs.metaflow.org)
- [Tutoriales interactivos](https://github.com/outerbounds/tutorials)
- [Outerbounds — la empresa detrás de Metaflow](https://outerbounds.com)